# Python Advanced — Coding Challenges

This notebook is a self-check arena for the advanced topics you have been studying: decorators, generators, context managers, properties, ordering, and abstract base classes. It contains ten hands-on problems, each one designed to be solved in about ten minutes — the pace interviewers use as a screening gate. The setup cell imports `numpy` and `pandas` and seeds the random generator, so the earlier exercises run in the same environment you will use in real data work. Every challenge is stated in the markdown before its solution; each solution ends with an `assert` so you can run it and instantly know whether you got it right.

The problems are deliberately varied. Challenges 1, 7, 8, and 4 spin the decorator and context-manager machinery: counting calls, retrying on a chosen exception, turning a *class* itself into a decorator, and rolling a dictionary back if a block raises. Challenges 3, 6, 9, and 2 work the lazy side and the read-only side of the language: a generator that yields only unseen items, a generator reading a list backwards without copying it, a running-average generator, and a computed `@property`. Challenges 5 and 10 close the list with ordering and contract enforcement: `functools.total_ordering` so custom objects are sortable, and an abstract base class that forces subclasses to implement `run()`.

Approach each challenge the same way: read the statement, mentally sketch the solution, write it in the cell, and let the assert judge you. The assert is the contract — if the cell prints `ok`, your implementation matches the required behaviour exactly. Where an idea is new (a class used as a decorator, say, or the `@contextmanager` protocol), the short theory cell that precedes the solution explains the mechanism and why it works. Use this notebook as a warm-up before interviews or as a review drill after the sessions.

## Setup

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
np.random.seed(0)
print('pandas', pd.__version__)

## Challenge 1

Write a decorator counting how many times a function is called.

## Challenge 1 — a counting decorator

This is the canonical "decorator with state" problem. A **decorator** is a function that receives a function and returns a wrapped version of it; here the wrapper must remember how many times the wrapped function ran. The trick is where the counter lives — it lives on the *wrapper function itself* as an attribute:

```python
w.calls += 1
w.calls = 0
```

Functions are objects, so they can carry attributes — `w.calls` is initialised to `0` once (at decoration time) and incremented on every invocation. Each call to `ping()` runs the closure `w`, bumps the counter, and returns the original result. `@functools.wraps(f)` copies the original function's `__name__` and docstring onto the wrapper so it does not lose its identity. The assert checks `ping.calls == 2` after two calls — a memory-efficient way to store the counter without touching any global or class. This pattern, sometimes called a memoized/measured decorator, is the first thing interviewers ask because it proves you understand closures, function attributes, and the `@` sugar all at once.

In [ ]:
import functools
def counted(f):
    @functools.wraps(f)
    def w(*a, **k):
        w.calls += 1
        return f(*a, **k)
    w.calls = 0
    return w

@counted
def ping(): pass

ping()
ping()
assert ping.calls == 2
print('ok')

## Challenge 2

Implement a class with a computed read-only property.

## Challenge 2 — computed read-only properties

A `@property` turns a method into an **attribute** that is computed on demand and, without a setter, is effectively read-only. The `Rect` class declares `area` as a method but exposes it as `rect.area`:

```python
@property
def area(self):
    return self.w * self.h
```

Every time you read `Rect(3, 4).area`, the method runs and recomputes `3 * 4`. Why not store the area as an attribute in `__init__`? Because it would change whenever `w` or `h` changes — stored state would go stale, while a computed property is always correct. Because no `@area.setter` is defined, assigning to `rect.area` raises `AttributeError`, which is exactly what "read-only" means. This is the foundation of controlled attribute access in Python: you write the attribute once as logic, and its users never know whether they are touching a field or a function.

In [ ]:
class Rect:
    def __init__(self, w, h): self.w, self.h = w, h
    @property
    def area(self): return self.w * self.h

assert Rect(3, 4).area == 12
print('ok')

## Challenge 3

Write a generator yielding only unique items from a stream.

## Challenge 3 — unique items from a stream

The `unique` function is a **generator** that yields the first occurrence of each distinct value it sees, using a `set` as its memory:

```python
seen = set()
for x in stream:
    if x not in seen:
        seen.add(x)
        yield x
```

Because it uses `yield`, it is lazy: it does not build a list of the result — it produces one value at a time as the consumer iterates, and it can run over an infinitely long stream without using infinite memory. The `seen` set is the price of uniqueness: membership tests in a set are O(1) on average, so the whole algorithm stays fast even on large inputs. The assert verifies order preservation too: `list(unique([1, 1, 2, 1, 3]))` gives `[1, 2, 3]` — first occurrences kept in arrival order, duplicates dropped. This is the "deduplicate without sorting" interview pattern, and the lazy generator version is the version you want in production pipelines.

In [ ]:
def unique(stream):
    seen = set()
    for x in stream:
        if x not in seen:
            seen.add(x)
            yield x

assert list(unique([1, 1, 2, 1, 3])) == [1, 2, 3]
print('ok')

## Challenge 4

Build a context manager that rolls back a dict on exception.

## Challenge 4 — a transaction context manager

The `@contextmanager` decorator lets you write a context manager out of a plain generator function. What makes it special is the `yield`: everything *before* the `yield` runs on entering the `with` block, the `yield` expression hands the value to the `as s:` variable, and everything *after* the `yield` runs on exiting — even if the block raised.

The rollback logic is classic pessimistic transaction semantics:

```python
backup = dict(d)      # snapshot before the block

try:
    yield d
except Exception:
    d.clear()
    d.update(backup)  # restore the snapshot on failure
```

The generator has a `try/except` around the `yield`, so an exception raised inside the `with` block is caught here — and for a dict, "roll back" means clearing it and restoring the backup. If the block succeeds, the changes stay. The assert confirms the contract: after a `with transaction(state) as s:` block that mutates `s` and then raises `ValueError`, `state` still equals `{'a': 1}` — the mutation was rolled back. This is the pattern behind database transactions and config rollbacks.

In [ ]:
from contextlib import contextmanager
@contextmanager
def transaction(d):
    backup = dict(d)
    try:
        yield d
    except Exception:
        d.clear()
        d.update(backup)

state = {'a': 1}
try:
    with transaction(state) as s:
        s['b'] = 2
        raise ValueError()
except ValueError:
    pass
assert state == {'a': 1}
print('ok')

## Challenge 5

Implement ordering so a list of objects can be sorted.

## Challenge 5 — sorting custom objects

Plain Python objects cannot be sorted by default — `<` on two `Score` instances would raise a `TypeError`. Sorting needs ordering comparisons (`<`, `<=`, `>`, `>=`), but it is tedious to write all four. `functools.total_ordering` solves it: implement `__eq__` (equality) and just **one** ordering comparison — here `__lt__` — and decorates the class so Python derives `<=`, `>`, `>=` from them:

```python
@functools.total_ordering
class Score:
    def __eq__(self, o): return self.v == o.v
    def __lt__(self, o): return self.v < o.v
```

The assert checks the sort itself: `sorted([Score(3), Score(1)])` puts the smaller score first, and taking `[0].v` yields `1`. The deeper point is that sorting in Python works through the comparison protocol — any class that implements it participates in `sorted()`, `min()`, `max()`, and `bisect` transparently. `__lt__` is the one comparison you really need; `__eq__` is required because Python 3 made comparisons and equality independent.

In [ ]:
import functools
@functools.total_ordering
class Score:
    def __init__(self, v): self.v = v
    def __eq__(self, o): return self.v == o.v
    def __lt__(self, o): return self.v < o.v

assert sorted([Score(3), Score(1)])[0].v == 1
print('ok')

## Challenge 6

Write a generator reading a list in reverse without copying it.

## Challenge 6 — reverse iteration without copying

The obvious way to reverse a sequence is `reversed(seq)` or a slice `seq[::-1]` — but both build a reversed *copy*. For a large list that is wasted memory. The `backwards` generator iterates over the indexes from the last to the first and yields each element, touching the original only:

```python
def backwards(seq):
    for i in range(len(seq) - 1, -1, -1):
        yield seq[i]
```

`range(len-1, -1, -1)` walks the indexes `2, 1, 0` for a list of three — the `-1` stop means "down to but not including index -1", which is the classic way to count down to zero. Each `yield` hands out `seq[i]` lazily, so the generator uses O(1) extra space and never allocates a reversed list. The assert confirms `list(backwards([1, 2, 3])) == [3, 2, 1]`. Interviewers ask for this to see whether you reach for an eager slice out of habit, or think about memory as you iterate.

In [ ]:
def backwards(seq):
    for i in range(len(seq) - 1, -1, -1):
        yield seq[i]

assert list(backwards([1, 2, 3])) == [3, 2, 1]
print('ok')

## Challenge 7

Create a decorator that retries on a specific exception type.

## Challenge 7 — a retry decorator with arguments

Sometimes a decorator needs its own configuration — here, *which* exception to retry and *how many times*. That requires the three-layer structure: the outermost function takes the arguments and returns the actual decorator, which takes the function and returns the wrapper:

```python
def retry(exc, times=3):
    def deco(f):
        def w(*a, **k):
            for _ in range(times):
                try:
                    return f(*a, **k)
                except exc:
                    continue
            raise exc('gave up')
        return w
    return deco
```

The wrapper attempts `f` up to `times` times. Every failure on the *specific* exception `exc` is swallowed and the loop tries again; if all attempts fail, it raises the same exception with a "gave up" message. The demo function fails on its first call (a counter starts at 0) and succeeds on the second, so `f()` returns `'ok'` and the assert passes. Selective retry matters in production: you retry transient failures (timeouts, rate limits) but let genuine bugs propagate — catching only `exc` and re-raising after the final attempt is exactly that discipline.

In [ ]:
import functools
def retry(exc, times=3):
    def deco(f):
        @functools.wraps(f)
        def w(*a, **k):
            for _ in range(times):
                try:
                    return f(*a, **k)
                except exc:
                    continue
            raise exc('gave up')
        return w
    return deco

n = {'c': 0}
@retry(ValueError)
def f():
    n['c'] += 1
    if n['c'] < 2:
        raise ValueError()
    return 'ok'

assert f() == 'ok'
print('ok')

## Challenge 8

Use a class as a decorator.

## Challenge 8 — a class as a decorator

Decorators do not have to be functions — any *callable* works, and a class is callable when it defines `__call__`. The `Logged` class stores the wrapped function at decoration time and answers calls through its dunder:

```python
class Logged:
    def __init__(self, f):
        self.f = f
        self.calls = 0
    def __call__(self, *a, **k):
        self.calls += 1
        return self.f(*a, **k)
```

`@Logged` above `def add(...)` is sugar for `add = Logged(add)`: Python constructs a `Logged` object passing `add` as `f`, so the instance remembers the original function and starts a counter at `0`. Calling `add(1, 2)` invokes `__call__`, which increments `self.calls` and delegates to the stored function. The big advantage over a function decorator is *state as attributes*: `self.calls` reads naturally as `add.calls`, and you can add more attributes — logs, thresholds, cache dictionaries — with ordinary `self.` assignments. The assert checks `add.calls == 1` after one call. Classes-as-decorators always impress in interviews because they show you understand that the `@` syntax only requires a callable.

In [ ]:
class Logged:
    def __init__(self, f):
        self.f = f
        self.calls = 0
    def __call__(self, *a, **k):
        self.calls += 1
        return self.f(*a, **k)

@Logged
def add(a, b): return a + b

add(1, 2)
assert add.calls == 1
print('ok')

## Challenge 9

Write a generator computing a running average lazily.

## Challenge 9 — a lazy running average

A running average asks: after each element, what is the mean of everything seen so far? The generator version keeps two tiny pieces of state — the running `total` and the count `n` — and yields after every update:

```python
def running_avg(stream):
    total, n = 0, 0
    for x in stream:
        total += x
        n += 1
        yield total / n
```

Why a generator and not a function that returns a list? Because the averages are *requests* — values are computed only when the consumer iterates, and the user can stop early without computing the rest of the stream. For the stream `[2, 4]`: after `2` the average is `2.0`, after `4` the running total is `6` over 2 items, giving `3.0` — the assert checks exactly `[2.0, 3.0]`. This is a classic streaming pattern for monitoring, sensor feeds, and online statistics: constant memory, one pass, no storage of the full history.

In [ ]:
def running_avg(stream):
    total, n = 0, 0
    for x in stream:
        total += x
        n += 1
        yield total / n

assert list(running_avg([2, 4])) == [2.0, 3.0]
print('ok')

## Challenge 10

Implement an abstract base class enforcing a method.

## Challenge 10 — abstract base classes enforce contracts

The `abc` module provides **abstract base classes**: classes that declare methods their subclasses *must* implement. `Base` inherits from `ABC` and marks `run` with `@abstractmethod`, so:

- `Impl(Base)` must define `run` — and the working subclass in the code returns `'ran'`, which the first assert verifies.
- You **cannot instantiate** `Base` itself: the attempt `Base()` raises `TypeError` with a message about the abstract method, which the `try/except` in the solution catches and prints as `abstract enforced`.

```python
class Base(ABC):
    @abstractmethod
    def run(self):
        raise NotImplementedError
```

The effect is a compile-time-like contract at runtime: designing a framework, you declare "every worker must implement `run`" and Python enforces it the moment someone tries to build an incomplete subclass. Note the difference between the two mechanisms: the `raise NotImplementedError` inside the abstract method is the body it would execute if a subclass forgot to override it, while the `ABC` machinery is what refuses instantiation. This is how large libraries keep their polymorphic contracts honest.

In [ ]:
from abc import ABC, abstractmethod
class Base(ABC):
    @abstractmethod
    def run(self):
        raise NotImplementedError

class Impl(Base):
    def run(self): return 'ran'

assert Impl().run() == 'ran'
try:
    Base()
except TypeError:
    print('abstract enforced')
print('ok')

## Summary

These ten challenges exercised the advanced toolkit end to end. On **decorators**, you counted calls with a function attribute (`challenge 1`), retried on a configurable exception (`challenge 7`), and turned a class into a decorator through `__call__` (`challenge 8`). On **context management**, you built `@contextmanager` rollback semantics for a dictionary (`challenge 4`). On **generators**, you filtered streams to unique items (`challenge 3`), read sequences backwards without copying (`challenge 6`), and computed running averages lazily (`challenge 9`). On the object model, you published a read-only computed `@property` (`challenge 2`) and achieved sorting with just `__eq__` and `__lt__` plus `functools.total_ordering` (`challenge 5`), while `challenge 10` showed how an `ABC` forces subclasses to implement a method. Every solution passed its assert, which is the contract that matters. Time yourself at ten minutes per problem; re-do the slower ones — that repetition, not reading, is what makes the advanced material stick.